# 01 — Linkage and analytical panels

**Purpose.** Standardize the four high-coverage Spotify snapshots, link the Billboard
universe to soundtrack film appearances, movie metadata, and Last.fm listeners, and build the
analytical panels used by every downstream model.

**Inputs.** `outputs/data/billboard_song_artist.csv` (notebook 00) and the preserved raw
snapshots and linkage inputs under `data/raw/`.

**Outputs.** `outputs/data/master_song_panel.csv` (song-level master with artist-catalog
covariates), `outputs/data/spotify_song_snapshot_panel.csv` (83,785 song-snapshot
observations), `outputs/data/standardized_lastfm_2017.csv`, `outputs/data/snapshot_time_respecting_exposures.csv`,
and the platform-coverage and movie-metadata coverage objects under `outputs/data/results/`.

**Unit of analysis.** Billboard song-artist records (master) and song-snapshot observations (panels).

**Linkage keys.** Two deterministic normalizations are used, both documented here:
(1) the *linkage key* (lower-case, featuring/with connectors removed, punctuation and accents
stripped) links Billboard records to Spotify snapshots, IMDb soundtrack rows, and Last.fm;
(2) the *catalog key* (punctuation-free alphanumeric string) groups songs into artist/band
catalogs and joins the movie-metadata and time-respecting exposure tables.


In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw_checksums.csv").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from IPython.display import display

from cdr import paths, build, util
from cdr.util import SNAPSHOTS, SNAPSHOT_LABEL, check, show_and_save_table, show_and_save_figure

paths.ensure_output_dirs()
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
RESULTS = paths.RESULTS

Matplotlib is building the font cache; this may take a moment.


## Master song panel: Spotify snapshots and film linkage

A Billboard record is **film-linked** if its normalized song/artist linkage key matches at
least one IMDb soundtrack appearance; `first_year` is the earliest linked film release year.
Records without a match are classified as having no observed film appearance (nonmatches can
reflect incomplete soundtrack coverage or limits of exact string linkage, not evidence the
song was never used in a film).


In [2]:
billboard = pd.read_csv(paths.OUT_DATA / "billboard_song_artist.csv", parse_dates=["chart_debut"], low_memory=False)
master = build.build_master_panel(billboard)
support = pd.DataFrame(
    [
        ("Billboard song-artist universe", len(master)),
        ("Film-linked records (exact IMDb soundtrack linkage)", int(master["film_linked"].sum())),
        ("Records with October 2016 Spotify popularity", int(master["pop2016"].notna().sum())),
        ("Records with July 2017 Spotify popularity", int(master["pop2017"].notna().sum())),
        ("Records with August 2022 Spotify popularity", int(master["pop2022"].notna().sum())),
        ("Records with August 2025 Spotify popularity", int(master["pop2025"].notna().sum())),
        ("Records with July 2017 Last.fm listeners", int(master["lastfm_listeners"].notna().sum())),
    ],
    columns=["stage", "n_records"],
)
show_and_save_table(support, RESULTS / "master_support_counts.csv")
check(len(master) == 32124, "master panel covers the full Billboard universe (32,124)")
check(int(master["film_linked"].sum()) == 4683, "film-linked Billboard song-artist records = 4,683")
check(int(master["lastfm_listeners"].notna().sum()) == 17988, "Last.fm July 2017 records with listeners = 17,988")


,stage,n_records
0,Billboard song-artist universe,32124
1,Film-linked records (exact IMDb soundtrack lin...,4683
2,Records with October 2016 Spotify popularity,17347
3,Records with July 2017 Spotify popularity,18189
4,Records with August 2022 Spotify popularity,22105
5,Records with August 2025 Spotify popularity,26144
6,Records with July 2017 Last.fm listeners,17988


PASS: master panel covers the full Billboard universe (32,124)
PASS: film-linked Billboard song-artist records = 4,683
PASS: Last.fm July 2017 records with listeners = 17,988


## Artist/band catalog keys and artist-level historical covariates

Catalogs are defined by the punctuation-free artist/band string. For each song we compute the
artist's other-song catalog size, total Billboard weeks excluding the focal song, mean weeks
and best peak among the other songs, a top-1%-by-total-weeks superstar indicator, and whether
the artist has at least one film-linked Billboard song other than the focal song.


In [3]:
master = build.add_artist_features(master)
n_artists = master["artist_key_alnum"].nunique()
print(f"Unique normalized artist/band catalogs: {n_artists:,}")
display(master[["artist_key_alnum", "artist_billboard_songs", "artist_total_billboard_weeks", "artist_superstar_top1pct", "artist_has_treated_song"]].drop_duplicates("artist_key_alnum").head(6))


Unique normalized artist/band catalogs: 11,080


,artist_key_alnum,artist_billboard_songs,artist_total_billboard_weeks,artist_superstar_top1pct,artist_has_treated_song
0,youngandrestless,1,15,0,0
1,neildiamond,52,553,1,1
2,theovationsfeaturinglouiswilliams,2,15,0,0
3,isaachayes,13,123,0,1
4,quincyjones,6,59,0,1
5,duaneeddyhistwangyguitarandtherebels,12,112,0,0


## Expanded movie metadata and the single coverage object

Movie attributes (worldwide box office, budget, genres) are linked through exact catalog keys
and exact internal movie IDs; no fuzzy title matching is used. This is the one authoritative
movie-metadata coverage/missingness object: all counts share the film-linked denominator.


In [4]:
master = build.add_expanded_movie_metadata(master)
coverage = build.movie_metadata_coverage(master)
show_and_save_table(coverage, RESULTS / "movie_metadata_coverage.csv")
check(int(coverage.loc[coverage["variable"].str.startswith("movie metadata"), "available"].iloc[0]) == 4682, "movie metadata available for 4,682 film-linked records")
check(int(coverage.loc[coverage["variable"].eq("worldwide box office"), "available"].iloc[0]) == 4272, "worldwide box office available for 4,272 film-linked records")
check(int(coverage.loc[coverage["variable"].eq("production budget"), "available"].iloc[0]) == 2581, "budget available for 2,581 film-linked records")


,variable,available,missing,n_film_linked_records,share_missing
0,movie metadata via exact song-artist / interna...,4682,1,4683,0.000214
1,worldwide box office,4272,411,4683,0.087764
2,production budget,2581,2102,4683,0.448858
3,genre,4682,1,4683,0.000214
4,IMDb votes / rating / country / language,0,4683,4683,1.000000


PASS: movie metadata available for 4,682 film-linked records
PASS: worldwide box office available for 4,272 film-linked records
PASS: budget available for 2,581 film-linked records


## High-coverage Spotify song-snapshot panel

The panel stacks the four high-coverage snapshots. `attention` is the platform 0-100
popularity index in its original scale; `age` is snapshot year minus chart-debut year;
`previous_popularity` is the song's popularity in the previous high-coverage snapshot
(available from July 2017 onward and used by baseline-attention matching designs).


In [5]:
panel = build.build_spotify_panel(master)
by_snap = panel.groupby("snapshot").agg(n_song_snapshots=("attention", "size"), n_film_linked=("film_linked", "sum")).reset_index()
display(by_snap)
print(f"Total nonmissing song-snapshot observations: {len(panel):,}")
print(f"Observations with valid nonnegative song age (expected-memory support): {int((panel['age'] >= 0).sum()):,}")
check(len(panel) == 83785, "Spotify high-coverage panel has 83,785 song-snapshot observations")
check(int((panel["age"] >= 0).sum()) == 83780, "83,780 observations have valid nonnegative song age")
panel.to_csv(paths.OUT_DATA / "spotify_song_snapshot_panel.csv", index=False)


,snapshot,n_song_snapshots,n_film_linked
0,2016-10,17347,3825
1,2017-07,18189,4133
2,2022-08,22105,4373
3,2025,26144,4616


Total nonmissing song-snapshot observations: 83,785
Observations with valid nonnegative song age (expected-memory support): 83,780
PASS: Spotify high-coverage panel has 83,785 song-snapshot observations
PASS: 83,780 observations have valid nonnegative song age


## Time-respecting film exposure by snapshot

For each song-snapshot, exposure counts only film appearances realized by the snapshot date:
the number of linked films released by the snapshot, the maximum worldwide box office among
those films, and the first film year observed by the snapshot. These variables cannot use
future film information.


In [6]:
exposures = build.build_time_respecting_exposures(master)
exposures.to_csv(paths.OUT_DATA / "snapshot_time_respecting_exposures.csv", index=False)
panel_tr = build.merge_time_respecting(panel[panel["age"] >= 0], exposures)
vis_support = panel_tr[
    panel_tr["film_linked_by_snapshot"].eq(1)
    & panel_tr["max_worldwide_boxoffice_by_snapshot"].notna()
    & (panel_tr["max_worldwide_boxoffice_by_snapshot"] > 0)
]
validation = pd.DataFrame(
    [
        ("film-linked song-snapshots with realized box office", len(vis_support)),
        ("unique songs in the realized box-office support", vis_support["song_artist_id"].nunique()),
        ("song-snapshots film-embedded by snapshot", int(panel_tr["film_linked_by_snapshot"].sum())),
    ],
    columns=["quantity", "n"],
)
show_and_save_table(validation, RESULTS / "time_respecting_exposure_validation.csv")
check(len(vis_support) == 15245, "time-respecting film-visibility support = 15,245 song-snapshots")
check(vis_support["song_artist_id"].nunique() == 4294, "time-respecting film-visibility support covers 4,294 unique songs")


,quantity,n
0,film-linked song-snapshots with realized box o...,15245
1,unique songs in the realized box-office support,4294
2,song-snapshots film-embedded by snapshot,16546


PASS: time-respecting film-visibility support = 15,245 song-snapshots
PASS: time-respecting film-visibility support covers 4,294 unique songs


## Film-timing summary

Distribution of first observed film years for film-linked records, relative to the four
Spotify snapshots (this is the timing support behind the pre/post and dormant designs).


In [7]:
film = master[master["film_linked"].eq(1)].copy()
timing = pd.DataFrame(
    [
        ("first film year before October 2016 snapshot", int((film["first_year"] <= 2016.80).sum())),
        ("first film year between Oct 2016 and Jul 2017", int(((film["first_year"] > 2016.80) & (film["first_year"] <= 2017.55)).sum())),
        ("first film year between Jul 2017 and Aug 2022", int(((film["first_year"] > 2017.55) & (film["first_year"] <= 2022.65)).sum())),
        ("first film year between Aug 2022 and Aug 2025", int(((film["first_year"] > 2022.65) & (film["first_year"] <= 2025.65)).sum())),
        ("first film year after August 2025 snapshot", int((film["first_year"] > 2025.65).sum())),
    ],
    columns=["window", "n_film_linked_records"],
)
show_and_save_table(timing, RESULTS / "film_timing_summary.csv")


,window,n_film_linked_records
0,first film year before October 2016 snapshot,4231
1,first film year between Oct 2016 and Jul 2017,77
2,first film year between Jul 2017 and Aug 2022,375
3,first film year between Aug 2022 and Aug 2025,0
4,first film year after August 2025 snapshot,0


,window,n_film_linked_records
0,first film year before October 2016 snapshot,4231
1,first film year between Oct 2016 and Jul 2017,77
2,first film year between Jul 2017 and Aug 2022,375
3,first film year between Aug 2022 and Aug 2025,0
4,first film year after August 2025 snapshot,0


## Platform coverage

The study's analytical support consists of the four high-coverage Spotify snapshots and the
single July 2017 Last.fm listener snapshot.


In [8]:
coverage_rows = []
for snap, year, col, label in SNAPSHOTS:
    g = panel[panel["snapshot"].eq(snap)]
    coverage_rows.append(
        {
            "platform": "Spotify",
            "snapshot": label,
            "records_with_outcome": len(g),
            "film_linked_with_outcome": int(g["film_linked"].sum()),
            "note": "high-coverage snapshot",
        }
    )
coverage_rows.append(
    {
        "platform": "Last.fm",
        "snapshot": "July 2017",
        "records_with_outcome": int(master["lastfm_listeners"].notna().sum()),
        "film_linked_with_outcome": int(master.loc[master["lastfm_listeners"].notna(), "film_linked"].sum()),
        "note": "single available snapshot; cumulative listener reach",
    }
)
platform_coverage = pd.DataFrame(coverage_rows)
show_and_save_table(platform_coverage, RESULTS / "platform_coverage.csv")
check(len(platform_coverage) == 5, "platform coverage lists exactly the four high-coverage Spotify snapshots and Last.fm")


,platform,snapshot,records_with_outcome,film_linked_with_outcome,note
0,Spotify,October 2016,17347,3825,high-coverage snapshot
1,Spotify,July 2017,18189,4133,high-coverage snapshot
2,Spotify,August 2022,22105,4373,high-coverage snapshot
3,Spotify,August 2025,26144,4616,high-coverage snapshot
4,Last.fm,July 2017,17988,4090,single available snapshot; cumulative listener...


PASS: platform coverage lists exactly the four high-coverage Spotify snapshots and Last.fm


## Save the authoritative analytical panels


In [9]:
master.to_csv(paths.OUT_DATA / "master_song_panel.csv", index=False)
lastfm = build.build_lastfm_support()
lastfm.to_csv(paths.OUT_DATA / "standardized_lastfm_2017.csv", index=False)
lastfm_panel = build.build_lastfm_panel(master)
lastfm_panel.to_csv(paths.OUT_DATA / "lastfm_song_panel.csv", index=False)
print("Saved master_song_panel.csv, spotify_song_snapshot_panel.csv, snapshot_time_respecting_exposures.csv,")
print("standardized_lastfm_2017.csv, and lastfm_song_panel.csv under outputs/data/.")
check(len(lastfm_panel) == 17988, "Last.fm analytical panel has 17,988 records")


Saved master_song_panel.csv, spotify_song_snapshot_panel.csv, snapshot_time_respecting_exposures.csv,
standardized_lastfm_2017.csv, and lastfm_song_panel.csv under outputs/data/.
PASS: Last.fm analytical panel has 17,988 records


## Paper artifacts for this section

Main Table 1 and the SI coverage tables are generated here, where the linkage and coverage result objects are produced. Notebook 06 only registers, validates, and syncs them into the manuscript.

In [10]:
# Paper-artifact helpers: shared figure style, LaTeX writers, result-object reader.
from matplotlib.gridspec import GridSpec

from cdr import tex
from cdr.tex import fmt_num, fmt_int, fmt_p, fmt_ci
from cdr.util import (ACCENT, BLUE, CHARCOAL, COV_LABEL, DARK_BLUE, GRAY, GREEN, GRID,
                      LIGHT_BLUE, LIGHT_GRAY, SNAP_SHORT, errorbarh, errorbarv,
                      panel_label, set_plot_style)

set_plot_style()
R = lambda name: pd.read_csv(RESULTS / name, low_memory=False)

In [11]:
support = R("master_support_counts.csv")
coverage = R("movie_metadata_coverage.csv")
def sup(stage):
    return int(support.loc[support["stage"].str.startswith(stage), "n_records"].iloc[0])
main_t1 = pd.DataFrame([
    ("Billboard Hot 100 universe", f"{sup('Billboard'):,} records", "Defines the population of previously successful songs and supplies chart-history measures"),
    ("Observed film reuse", f"{sup('Film-linked'):,} records", "Identifies Billboard records with at least one observed soundtrack appearance and their first observed film year"),
    ("Spotify attention", "83,785 song-snapshots", "Four high-coverage snapshots (2016, 2017, 2022, 2025) used for broad and temporal attention comparisons"),
    ("Expected-attention support", "83,780 song-snapshots", "Valid nonnegative song age, covering 28,310 distinct Billboard song-artist records, used to estimate age- and snapshot-specific expected attention"),
    ("Last.fm listener reach", f"{sup('Records with July 2017 Last.fm'):,} records", "July 2017 cumulative listeners, providing a complementary measure of listener reach"),
    ("Expanded film metadata", f"{int(coverage.loc[coverage['variable'].str.startswith('movie metadata'), 'available'].iloc[0]):,} film-linked records", "Movie-level information used to characterize repeated embedding and film context"),
    ("Worldwide box office", f"{int(coverage.loc[coverage['variable'].eq('worldwide box office'), 'available'].iloc[0]):,} film-linked records", "Recorded worldwide grosses used as a proxy for film-level visibility"),
], columns=["Data component", "Coverage", "Analytical role"])
display(main_t1)
table_path = paths.OUT_TABLES_MAIN / "data_audit_table.tex"
tex.write_table(table_path,
    "Data foundation for distinguishing prior visibility from attention renewal.", "tab:data-audit",
    ["Data component", "Coverage", "Analytical role"],
    main_t1.values.tolist(),
    note="Counts describe the broad linked sources before temporal and matching restrictions. Detailed platform coverage, linkage procedures, and movie-metadata missingness are reported in the SI Appendix, Data Construction and Platform Coverage.",
    col_spec=r"@{}>{\raggedright\arraybackslash}p{0.24\linewidth}>{\raggedright\arraybackslash}p{0.17\linewidth}>{\raggedright\arraybackslash}p{0.53\linewidth}@{}")
table_tex = table_path.read_text(encoding="utf-8").replace(r"\begin{table}[!htbp]", r"\begin{table}[H]", 1)
table_path.write_text(table_tex, encoding="utf-8")


,Data component,Coverage,Analytical role
0,Billboard Hot 100 universe,"32,124 records",Defines the population of previously successfu...
1,Observed film reuse,"4,683 records",Identifies Billboard records with at least one...
2,Spotify attention,"83,785 song-snapshots","Four high-coverage snapshots (2016, 2017, 2022..."
3,Expected-attention support,"83,780 song-snapshots","Valid nonnegative song age, covering 28,310 di..."
4,Last.fm listener reach,"17,988 records","July 2017 cumulative listeners, providing a co..."
5,Expanded film metadata,"4,682 film-linked records",Movie-level information used to characterize r...
6,Worldwide box office,"4,272 film-linked records",Recorded worldwide grosses used as a proxy for...


**Data foundation for distinguishing prior visibility from attention renewal.** (`tab:data-audit`)

,Data component,Coverage,Analytical role
0,Billboard Hot 100 universe,"32,124 records",Defines the population of previously successfu...
1,Observed film reuse,"4,683 records",Identifies Billboard records with at least one...
2,Spotify attention,"83,785 song-snapshots","Four high-coverage snapshots (2016, 2017, 2022..."
3,Expected-attention support,"83,780 song-snapshots","Valid nonnegative song age, covering 28,310 di..."
4,Last.fm listener reach,"17,988 records","July 2017 cumulative listeners, providing a co..."
5,Expanded film metadata,"4,682 film-linked records",Movie-level information used to characterize r...
6,Worldwide box office,"4,272 film-linked records",Recorded worldwide grosses used as a proxy for...


1732

In [12]:
# S: platform coverage (includes excluded January 2016 audit row) -- C2, C4
pc = R("platform_coverage.csv")
tex.write_table(paths.OUT_TABLES_SI / "table_si_platform_coverage.tex",
    "Platform and snapshot coverage.", "tab:si-platform-coverage",
    ["Platform", "Snapshot", "Records", "Film-linked", "Note"],
    [[r["platform"], r["snapshot"], fmt_int(r["records_with_outcome"]), fmt_int(r["film_linked_with_outcome"]), r["note"]] for _, r in pc.iterrows()],
    note="Records are Billboard song-artist records with a nonmissing outcome in each snapshot after exact normalized song-artist linkage. The Spotify analyses use the four high-coverage snapshots; Last.fm provides the single July 2017 listener snapshot.",
    col_spec=r">{\raggedright\arraybackslash}p{0.08\linewidth}>{\raggedright\arraybackslash}p{0.13\linewidth}>{\centering\arraybackslash}p{0.078\linewidth}>{\centering\arraybackslash}p{0.08\linewidth}>{\raggedright\arraybackslash}p{0.41\linewidth}",
    size=r"\footnotesize", stretch=1.05)

# S: movie metadata coverage -- C3
mc_ = R("movie_metadata_coverage.csv")
tex.write_table(paths.OUT_TABLES_SI / "table_si_movie_metadata_coverage.tex",
    "Movie-metadata coverage and missingness among film-linked records.", "tab:si-movie-metadata-coverage",
    ["Variable", "Available", "Missing", r"\makecell[b]{Film-linked\\ records}", r"\makecell[b]{Share\\ missing}"],
    [[r["variable"], fmt_int(r["available"]), fmt_int(r["missing"]), fmt_int(r["n_film_linked_records"]), fmt_num(r["share_missing"], 3)] for _, r in mc_.iterrows()],
    note="Movie attributes are linked through exact catalog keys and exact internal movie IDs; no fuzzy title matching is used and no values are imputed. All rows share the film-linked denominator. Analyses using box office, budget, or genre run on the corresponding supported subsets.",
    col_spec=r">{\raggedright\arraybackslash}p{0.34\linewidth}>{\centering\arraybackslash}p{0.095\linewidth}>{\centering\arraybackslash}p{0.085\linewidth}>{\centering\arraybackslash}p{0.115\linewidth}>{\centering\arraybackslash}p{0.09\linewidth}", stretch=1.05, escape_cells=False)

**Platform and snapshot coverage.** (`tab:si-platform-coverage`)

,Platform,Snapshot,Records,Film-linked,Note
0,Spotify,October 2016,"17,347","3,825",high-coverage snapshot
1,Spotify,July 2017,"18,189","4,133",high-coverage snapshot
2,Spotify,August 2022,"22,105","4,373",high-coverage snapshot
3,Spotify,August 2025,"26,144","4,616",high-coverage snapshot
4,Last.fm,July 2017,"17,988","4,090",single available snapshot; cumulative listener...


**Movie-metadata coverage and missingness among film-linked records.** (`tab:si-movie-metadata-coverage`)

,Variable,Available,Missing,Film-linked records,Share missing
0,movie metadata via exact song-artist / interna...,"4,682",1,"4,683",0.000
1,worldwide box office,"4,272",411,"4,683",0.088
2,production budget,"2,581","2,102","4,683",0.449
3,genre,"4,682",1,"4,683",0.000
4,IMDb votes / rating / country / language,0,"4,683","4,683",1.000
